# CatBoost с историческими аналогами маршрутов
Сравниваем остаточную модель, которая видит распределение будущих значений после похожих исторических утр. Соседей ищем отдельно для маршрута по причинным изменениям цели и статусов. Похожие дни определяются без знания их целевых значений, а сами дни и их цели строго предшествуют прогнозу. Модель и её вес выбираются на трёх последовательных неделях.


In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from catboost import CatBoostRegressor
from IPython.display import display

ROOT = Path(os.environ.get('BLEND_PACKAGE', '/root/experiments/session6_blend_server'))
OOF_PATH = None
TEST_PATH = None
OUT = ROOT / 'output_route_analogs_gpu'
OUT.mkdir(parents=True, exist_ok=True)

BASES = ['session6', 'rf', 'cat_rich', 'xgb_rich', 'xgb_clean', 'profile28']
GROUP_KEYS = ['cutoff', 'horizon']
KEYS = GROUP_KEYS + ['route_id']
COLUMNS = KEYS + BASES + ['y', 'level24', 'current', 'delta1h', 'delta2h', 'network_current']
THREADS = 2
SEED = 42

def read_panel(path, name, start, end):
    archived = ROOT / 'blend_inputs' / 'blend_best_output' / name
    if path is None and archived.is_file():
        path = archived
    if path is None:
        candidates = [p for p in ROOT.rglob(name) if OUT not in p.parents
                      and set(COLUMNS).issubset(pq.ParquetFile(p).schema.names)]
        if len(candidates) != 1:
            raise ValueError(f'Укажите путь явно для {name}: {candidates}')
        path = candidates[0]
    frame = pd.read_parquet(path, columns=COLUMNS)
    frame['cutoff'] = pd.to_datetime(frame.cutoff)
    expected = pd.date_range(start, end, inclusive='left') + pd.Timedelta(hours=10, minutes=30)
    actual = pd.DatetimeIndex(frame.cutoff.unique()).sort_values()
    counts = frame.groupby(GROUP_KEYS).size()
    if (not actual.equals(expected) or frame.duplicated(KEYS).any()
            or set(frame.horizon.unique()) != set(range(1, 9))
            or len(counts) != len(expected)*8 or not counts.eq(1000).all()
            or not np.isfinite(frame[COLUMNS[3:]].to_numpy()).all()
            or frame.y.lt(0).any() or frame.session6.lt(0).any()):
        raise ValueError(f'Некорректная панель: {path}')
    print(path, len(frame))
    return frame.reset_index(drop=True)

DATA = ROOT/'blend_inputs/session6'
BASE_RUN = Path(os.environ.get('BLEND_BASE_RUN',str(ROOT/'output/blend_submit_948a52023607')))
PARENT = Path(os.environ.get('OVERLAP_H1_PARENT',str(ROOT/'output_overlap_composite/submission_overlap_composite_full_history.csv')))
RUN_TEST = BASE_RUN/'train_val/test_predictions.parquet'
RUN_FUTURE = BASE_RUN/'full_history/future_predictions.parquet'
STEP = pd.Timedelta(minutes=30)
MIN_TEST_GAIN = .00015


In [ ]:
def with_volume(frame, values, anchor=None):
    result = np.maximum(np.asarray(values, dtype=np.float64), 0)
    reference = frame.session6.to_numpy(dtype=np.float64) if anchor is None else np.asarray(anchor, dtype=np.float64)
    totals = frame[GROUP_KEYS].assign(pred=result, reference=reference).groupby(GROUP_KEYS, sort=False).transform('sum')
    reference_total = totals.reference.to_numpy()
    prediction_total = totals.pred.to_numpy()
    out = np.divide(result * reference_total, prediction_total, out=reference.copy(), where=prediction_total > 0)
    predicted = frame[GROUP_KEYS].assign(value=out).groupby(GROUP_KEYS, sort=False).value.sum()
    expected = frame[GROUP_KEYS].assign(value=reference).groupby(GROUP_KEYS, sort=False).value.sum()
    if not np.isfinite(out).all() or not np.allclose(predicted, expected, rtol=1e-10, atol=1e-5):
        raise ValueError('Не удалось сохранить заданный объём')
    return out

def shape_matrix(frame):
    return np.column_stack([with_volume(frame, frame[name]) for name in BASES])

def fit_weights(x, y):
    y = np.asarray(y, dtype=np.float64)
    denominator = y.sum()
    if denominator <= 0:
        raise ValueError('Нулевая сумма фактов для подбора весов')
    weights = np.zeros(x.shape[1])
    weights[0] = 1
    prediction = x[:, 0].copy()
    best = np.abs(prediction - y).sum() / denominator
    for step in (.2, .05):
        for _ in range(20):
            choice = None
            for source in np.flatnonzero(weights >= step - 1e-10):
                for destination in range(len(weights)):
                    if source == destination:
                        continue
                    amount = min(step, weights[source])
                    trial = prediction + amount * (x[:, destination] - x[:, source])
                    loss = np.abs(trial - y).sum() / denominator
                    if loss < best - 1e-9:
                        best = loss
                        choice = source, destination, amount, trial
            if choice is None:
                break
            source, destination, amount, prediction = choice
            weights[source] -= amount
            weights[destination] += amount
    return weights

def meta_features(frame, matrix, baseline):
    level = np.maximum(frame.level24.to_numpy(dtype=np.float64), 1)
    context = pd.DataFrame(dict(
        log_level=np.log1p(level), current=frame.current.to_numpy() / level,
        delta1h=frame.delta1h.to_numpy() / level, delta2h=frame.delta2h.to_numpy() / level,
        log_network=np.log1p(frame.network_current.to_numpy()),
        horizon=frame.horizon.to_numpy(), dayofweek=frame.cutoff.dt.day_of_week.to_numpy(),
        shape_disagreement=matrix.std(axis=1) / level, blend=baseline / level))
    for index, name in enumerate(BASES):
        context[f'{name}_at_anchor_volume'] = matrix[:, index] / level
        context[f'{name}_minus_session6'] = (matrix[:, index] - matrix[:, 0]) / level
    return context.clip(-50, 50).to_numpy(dtype=np.float32), level

def metric(y, prediction):
    actual = np.asarray(y, dtype=np.float64).ravel()
    predicted = np.asarray(prediction, dtype=np.float64).ravel()
    volume = actual.sum()
    if actual.shape != predicted.shape or not np.isfinite(actual).all() or not np.isfinite(predicted).all():
        raise ValueError('Некорректный вход метрики')
    if volume == 0:
        return dict(rows=len(actual), WAPE=np.nan, relative_bias=np.nan, score=np.nan)
    error = predicted - actual
    wape = np.abs(error).sum() / volume
    bias = error.sum() / volume
    return dict(rows=len(actual), WAPE=wape, relative_bias=bias, score=wape + abs(bias))

def fit_meta(frame):
    matrix = shape_matrix(frame)
    weights = fit_weights(matrix, frame.y)
    baseline = matrix @ weights
    features, level = meta_features(frame, matrix, baseline)
    model = CatBoostRegressor(depth=4, iterations=220, loss_function='MAE', learning_rate=.035,
                              l2_leaf_reg=20, random_seed=SEED, verbose=False, allow_writing_files=False,
                              task_type='CPU', thread_count=THREADS)
    model.fit(features, (frame.y.to_numpy(dtype=float)-baseline)/level, sample_weight=level/level.mean())
    return model, weights

def predict_meta(frame, fitted):
    model, weights = fitted
    matrix = shape_matrix(frame)
    baseline = matrix @ weights
    features, level = meta_features(frame, matrix, baseline)
    return with_volume(frame, baseline + np.clip(model.predict(features), -.5, .5)*level)

def score_mix(frame, shape, weight):
    prediction = with_volume(frame, (1-weight)*frame.session6.to_numpy() + weight*shape)
    daily = pd.DataFrame([dict(day=str(day.date()), **metric(group.y, prediction[group.index]))
                          for day, group in frame.groupby('cutoff')])
    summary = dict(**metric(frame.y, prediction), daily_mean=daily.score.mean(),
                   daily_p90=daily.score.quantile(.9), daily_worst=daily.score.max())
    return summary, daily, prediction


In [ ]:
def load_target(path):
    raw = pd.read_parquet(path,columns=['timestamp','route_id','target_1h'])
    panel = raw.pivot(index='timestamp',columns='route_id',values='target_1h').sort_index().sort_index(axis=1)
    if (panel.shape[1]!=1000 or not panel.index.equals(pd.date_range(panel.index[0],panel.index[-1],freq=STEP))
            or not np.isfinite(panel.to_numpy()).all() or panel.lt(0).any().any()):
        raise ValueError('Некорректная история цели')
    return panel

def overlap_features(frame,level):
    x = np.zeros((len(frame),5),dtype=np.float32)
    mask = frame.horizon.eq(1).to_numpy()
    if not mask.any():
        return x
    selected = frame.loc[mask]
    positions = TARGET_PANEL.index.get_indexer(pd.DatetimeIndex(selected.cutoff))
    routes = TARGET_PANEL.columns.get_indexer(selected.route_id)
    if np.any(positions<7) or np.any(routes<0):
        raise ValueError('Не хватает причинной истории h1')
    seq = TARGET_PANEL.to_numpy(dtype=np.float64,copy=False)[positions[:,None]-np.arange(7,-1,-1),routes[:,None]]
    low = np.zeros(len(seq))
    high = seq[:,0].copy()
    valid = np.ones(len(seq),dtype=bool)
    for value in seq[:,1:].T:
        low,high = np.maximum(0,value-high),np.minimum(value,value-low)
        valid &= low<=high+1e-6
    low = np.where(valid,low,0.)
    high = np.where(valid,np.maximum(high,low),seq[:,-1])
    midpoint = (high+low)/2
    x[mask] = np.column_stack([low,high-low,midpoint,
                               midpoint-np.divide(seq[:,-1],2),valid*level[mask]])/level[mask,None]
    return x

def design(frame,matrix,baseline,augmented):
    context,level = meta_features(frame,matrix,baseline)
    if augmented:
        context = np.column_stack([context,overlap_features(frame,level)])
    return np.clip(context,-50,50).astype(np.float32,copy=False),level

def train_pair(frame):
    matrix = shape_matrix(frame)
    weights = fit_weights(matrix,frame.y)
    baseline = matrix @ weights
    models = {}
    for name,augmented in [('control',False),('overlap_h1',True)]:
        features,level = design(frame,matrix,baseline,augmented)
        model = CatBoostRegressor(depth=4,iterations=220,loss_function='MAE',learning_rate=.035,
                                  l2_leaf_reg=20,random_seed=SEED,verbose=False,allow_writing_files=False,
                                  task_type='CPU',thread_count=4)
        model.fit(features,(frame.y.to_numpy(dtype=float)-baseline)/level,sample_weight=level/level.mean())
        models[name] = model
    return weights,models

def predict_pair(frame,fitted):
    weights,models = fitted
    matrix = shape_matrix(frame)
    baseline = matrix @ weights
    result = {}
    for name,model in models.items():
        features,level = design(frame,matrix,baseline,name=='overlap_h1')
        residual = np.clip(model.predict(features),-.5,.5)*level
        result[name] = with_volume(frame,baseline+residual)
    if 'control' in result and 'overlap_h1' in result:
        result['overlap_h1'] = np.where(frame.horizon.eq(1),result['overlap_h1'],result['control'])
    return result

def summarize(frame,forecasts,fold,period):
    rows = []
    h1 = frame.horizon.eq(1).to_numpy()
    for name,prediction in forecasts.items():
        first = metric(frame.y.to_numpy()[h1],prediction[h1])
        daily = [metric(g.y,prediction[g.index])['score'] for _,g in frame.groupby('cutoff')]
        rows.append(dict(fold=fold,period=period,variant=name,**metric(frame.y,prediction),
                         h1_WAPE=first['WAPE'],h1_score=first['score'],
                         daily_mean=float(np.mean(daily)),daily_p90=float(np.quantile(daily,.9)),
                         daily_worst=float(np.max(daily))))
    return rows


RECIPES = {'analog_shallow': (4,360,'GPU'), 'analog_deep': (6,480,'GPU')}
MIXES = (.35,.65,1.)
SIGNALS = ['target_1h','status_1','status_3','status_5','status_6']

def read_analog_history(path):
    raw = pd.read_parquet(path,columns=['timestamp','route_id',*SIGNALS]).sort_values(['timestamp','route_id'])
    routes = pd.Index(raw.route_id.unique()).sort_values()
    stamps = pd.DatetimeIndex(raw.timestamp.unique()).sort_values()
    if (len(routes)!=1000 or len(raw)!=len(stamps)*len(routes)
            or raw.duplicated(['timestamp','route_id']).any() or not stamps.equals(TARGET_PANEL.index)):
        raise ValueError('Неполная история маршрутов')
    values = raw[SIGNALS].to_numpy(dtype=np.float32).reshape(len(stamps),len(routes),len(SIGNALS))
    return routes,values

def build_analogs(cutoffs):
    dates = TARGET_PANEL.index.get_indexer(pd.DatetimeIndex(cutoffs))
    if np.any(dates<48*42+8):
        raise ValueError('Недостаточно исторических дней')
    features = []
    for position in dates:
        distances = []
        past = position-48*np.arange(1,43)
        for signal,lag,weight in [(0,0,1.),(0,2,1.),(0,4,.7),(0,8,.5),
                                  (1,0,.45),(2,0,.45),(3,0,.6),(4,0,.6),
                                  (3,4,.35),(4,4,.35)]:
            candidates = np.log1p(ANALOG_VALUES[past-lag,:,signal])
            current = np.log1p(ANALOG_VALUES[position-lag,:,signal])
            scale = np.maximum(np.std(candidates,axis=0),.12)
            distances.append(weight*np.minimum(((candidates-current)/scale)**2,16.))
        distance = np.sum(distances,axis=0)/len(distances)
        days = TARGET_PANEL.index[past].day_of_week.to_numpy()
        query_day = TARGET_PANEL.index[position].day_of_week
        distance += .25*((days>=5)!=(query_day>=5))[:,None]
        distance += .15*(days!=query_day)[:,None]
        distance += .16*np.arange(42)[:,None]/41
        indices = np.argsort(distance,axis=0,kind='stable')[:10]
        nearest_distances = np.take_along_axis(distance,indices,axis=0)
        outcomes = ANALOG_VALUES[past[:,None]+np.arange(1,9)[None,:],:,0]
        outcomes = np.take_along_axis(outcomes,indices[:,None,:],axis=0)
        weights = np.exp(-(nearest_distances-nearest_distances[0])/.5)
        weighted = (outcomes*weights[:,None,:]).sum(axis=0)/weights.sum(axis=0)[None,:]
        q25,median,q75 = np.quantile(outcomes,[.25,.5,.75],axis=0)
        tight = outcomes[:3].mean(axis=0)
        general = np.median(ANALOG_VALUES[past[:,None]+np.arange(1,9)[None,:],:,0],axis=0)
        feature = np.stack([weighted,median,q75-q25,tight,general,weighted-general,
                            np.broadcast_to(nearest_distances[0],weighted.shape)],axis=-1)
        features.append(feature.astype(np.float32))
    return np.stack(features)

def temporal_inputs(frame,matrix,baseline):
    context,level = meta_features(frame,matrix,baseline)
    cutoffs = pd.DatetimeIndex(frame.cutoff.unique()).sort_values()
    missing = cutoffs.difference(ANALOG_CACHE.keys())
    if len(missing):
        ANALOG_CACHE.update(zip(missing,build_analogs(missing)))
    analog = np.stack([ANALOG_CACHE[cutoff] for cutoff in cutoffs])
    route_positions = ANALOG_ROUTES.get_indexer(frame.route_id)
    if (route_positions<0).any():
        raise ValueError('Неизвестный маршрут')
    rows = analog[cutoffs.get_indexer(frame.cutoff),frame.horizon.to_numpy(dtype=int)-1,route_positions]
    extras = np.column_stack([rows[:,:6]/level[:,None],rows[:,6],overlap_features(frame,level)])
    return np.column_stack([context,extras]).clip(-50,50).astype(np.float32),level

def fit_challengers(frame,names):
    control = train_pair(frame)
    if not names:
        return control,{}
    matrix = shape_matrix(frame)
    baseline = matrix @ control[0]
    reference = predict_pair(frame,control)['overlap_h1']
    features,level = temporal_inputs(frame,matrix,baseline)
    models = {}
    for name in names:
        depth,iterations,device = RECIPES[name]
        model = CatBoostRegressor(depth=depth,iterations=iterations,task_type=device,devices='0',
                                  loss_function='MAE',learning_rate=.035,l2_leaf_reg=25,
                                  random_seed=SEED,verbose=False,allow_writing_files=False,thread_count=4)
        model.fit(features,(frame.y.to_numpy()-reference)/level,sample_weight=level/level.mean())
        models[name] = model
        print('Обучен',name,len(frame),flush=True)
    return control,models

def predict_challengers(frame,fitted):
    control,models = fitted
    reference = predict_pair(frame,control)['overlap_h1']
    forecasts = {'overlap_h1':reference}
    if not models:
        return forecasts
    matrix = shape_matrix(frame)
    features,level = temporal_inputs(frame,matrix,matrix @ control[0])
    for name,model in models.items():
        expert = with_volume(frame,reference+np.clip(model.predict(features),-.5,.5)*level)
        for weight in MIXES:
            forecasts[f'{name}@{weight}'] = reference+weight*(expert-reference)
    return forecasts


In [ ]:

TARGET_PANEL = load_target(DATA/'train_solo_track.parquet')
ANALOG_ROUTES,ANALOG_VALUES = read_analog_history(DATA/'train_solo_track.parquet')
ANALOG_CACHE = {}
oof = read_panel(OOF_PATH,'base_oof.parquet','2025-09-20','2025-10-18')
rows = []
for start,end in [('2025-09-27','2025-10-04'),('2025-10-04','2025-10-11'),('2025-10-11','2025-10-18')]:
    train = oof.loc[oof.cutoff+oof.horizon*STEP < pd.Timestamp(start)].reset_index(drop=True)
    val = oof.loc[oof.cutoff.between(start,end,inclusive='left')].reset_index(drop=True)
    print('Валидация',start,flush=True)
    fitted = fit_challengers(train,RECIPES)
    rows.extend(summarize(train,predict_challengers(train,fitted),start,'train_oof'))
    rows.extend(summarize(val,predict_challengers(val,fitted),start,'val'))
    display(pd.DataFrame(rows).loc[lambda table: table.fold.eq(start)])
validation = pd.DataFrame(rows)
validation.to_csv(OUT/'validation_metrics.csv',index=False)
scores = validation.loc[validation.period.eq('val')].pivot(index='fold',columns='variant',values='daily_mean')
overall = validation.loc[validation.period.eq('val')].pivot(index='fold',columns='variant',values='score')
gains = -scores.sub(scores.overlap_h1,axis=0)
eligible = (gains.gt(0).all() & overall.le(overall.overlap_h1,axis=0).all())
rank = gains.loc[:,eligible].mean().sort_values(ascending=False)
chosen = rank.index[0] if len(rank) else None
(OUT/'frozen_recipe.json').write_text(json.dumps(dict(chosen=chosen,test_used_for_tuning=False,min_test_gain=MIN_TEST_GAIN),indent=2))
print('Зафиксированный кандидат',chosen,flush=True)


In [ ]:

test = read_panel(RUN_TEST,'test_predictions.parquet','2025-10-18','2025-11-01')
names = [chosen.split('@')[0]] if chosen else []
fitted = fit_challengers(oof,names)
for period,frame in [('train_oof',oof),('test',test)]:
    predictions = predict_challengers(frame,fitted)
    selected = {name:predictions[name] for name in ('overlap_h1',chosen) if name is not None}
    rows.extend(summarize(frame,selected,'2025-10-18',period))
metrics = pd.DataFrame(rows)
metrics.to_csv(OUT/'metrics.csv',index=False)
display(metrics.loc[metrics.fold.eq('2025-10-18')])
evaluation = metrics.loc[metrics.period.eq('test')].set_index('variant')
gain = float(evaluation.loc['overlap_h1','daily_mean']-evaluation.loc[chosen,'daily_mean']) if chosen else None
promoted = bool(chosen and gain>=MIN_TEST_GAIN and evaluation.loc[chosen,'score']<evaluation.loc['overlap_h1','score'])
(OUT/'decision.json').write_text(json.dumps(dict(chosen=chosen,promoted=promoted,test_daily_gain=gain,test_before_full_refit=True),indent=2))
print('Решение',promoted,'выигрыш',gain,flush=True)


In [ ]:

if promoted:
    future = pd.read_parquet(RUN_FUTURE,columns=[c for c in COLUMNS if c!='y'])
    future['cutoff'] = pd.to_datetime(future.cutoff)
    request = pd.read_parquet(DATA/'test_solo_track.parquet',columns=['id','route_id','timestamp'])
    parent = pd.read_csv(PARENT)
    cutoff = request.timestamp.min()-STEP
    history = pd.concat([oof,test],ignore_index=True)
    if (len(parent)!=8000 or len(request)!=8000 or len(future)!=8000 or not parent.id.is_unique
            or not request.id.is_unique or set(parent.id)!=set(request.id) or future.duplicated(KEYS).any()
            or not future.cutoff.eq(cutoff).all() or TARGET_PANEL.index[-1]!=cutoff
            or not (history.cutoff+history.horizon*STEP<=cutoff).all()):
        raise ValueError('Несовместимые данные для финального refit')
    fitted_final = fit_challengers(history,names)
    summary = summarize(history,{chosen:predict_challengers(history,fitted_final)[chosen]},'full_history','train_oof')
    display(pd.DataFrame(summary))
    pd.DataFrame(summary).to_csv(OUT/'refit_train_metrics.csv',index=False)
    forecasts = predict_challengers(future,fitted_final)
    future['correction'] = forecasts[chosen]-forecasts['overlap_h1']
    future['timestamp'] = future.cutoff+future.horizon*STEP
    query = request.merge(parent,on='id',validate='one_to_one',sort=False).merge(
        future[['route_id','timestamp','correction']],on=['route_id','timestamp'],validate='one_to_one',how='left',sort=False)
    if query.correction.isna().any() or not query.id.equals(request.id):
        raise ValueError('Ключи конкурса не совпали')
    query['cutoff'] = cutoff
    query['horizon'] = ((query.timestamp-cutoff)/STEP).astype(int)
    mask = np.ones(len(query),dtype=bool)
    prediction = query.y_pred.to_numpy(dtype=float,copy=True)
    prediction[mask] = with_volume(query.loc[mask],prediction[mask]+query.loc[mask,'correction'].to_numpy(),anchor=prediction[mask])
    if not np.isfinite(prediction).all() or (prediction<0).any():
        raise ValueError('Некорректные конкурсные прогнозы')
    destination = OUT/'submission_route_analogs_full_history.csv'
    query[['id']].assign(y_pred=prediction).to_csv(destination.with_suffix('.partial'),index=False)
    destination.with_suffix('.partial').replace(destination)
    fitted_final[1][names[0]].save_model(str(OUT/'route_analogs_full_history.cbm'))
    print('Сабмит',destination,flush=True)
print('Результаты',OUT,flush=True)
